<a href="https://colab.research.google.com/github/marcoastupp/ai-flux-training/blob/main/LivePortrait_Animator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. INSTALAÇÃO E COMPILAÇÃO DO LIVEPORTRAIT (COLAB T4)
# ==============================================================================
%cd /content
!git clone -b dev https://github.com/camenduru/LivePortrait
%cd /content/LivePortrait

!pip install -q tyro==0.8.5 onnxruntime-gpu onnx gradio==4.36.1 colorama huggingface_hub

# Compila extensão Cython para malha facial 3D
%cd /content/LivePortrait/src/utils/dependencies/insightface/thirdparty/face3d/mesh/cython
!python setup.py build_ext --inplace

# ==============================================================================
# 2. DOWNLOAD SEGURO DOS PESOS E PATCH DO PYTORCH
# ==============================================================================
import os
import shutil
import sys
import torch
import torch.serialization
from huggingface_hub import snapshot_download
from google.colab import drive

# Monta o Google Drive (onde ficarão a imagem base e o vídeo guia)
drive.mount('/content/drive')

weights_path = '/content/LivePortrait/pretrained_weights'
if os.path.exists(weights_path):
    shutil.rmtree(weights_path)

insightface_path = os.path.expanduser('~/.insightface/models/antelopev2')
if os.path.exists(insightface_path):
    shutil.rmtree(insightface_path)

print("⬇️ Baixando pesos oficiais do LivePortrait via Hugging Face Hub...")
snapshot_download(
    repo_id="camenduru/LivePortrait",
    local_dir=weights_path,
    local_dir_use_symlinks=False
)
print("✅ Pesos baixados com sucesso!")

# Patch para compatibilidade com PyTorch 2.6+
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "0"
_real_torch_load = torch.serialization.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _real_torch_load(*args, **kwargs)
torch.load = _patched_load

# ==============================================================================
# 3. EXECUÇÃO DA ANIMAÇÃO (IMAGEM BASE -> VÍDEO GUIA)
# ==============================================================================
os.chdir('/content/LivePortrait')
if '/content/LivePortrait' not in sys.path:
    sys.path.append('/content/LivePortrait')

# Defina aqui os caminhos da foto gerada no Flux (-s) e do vídeo de expressão (-d)
IMAGEM_BASE = '/content/drive/MyDrive/Wav2Lip/inicial_so_max_.png'
VIDEO_GUIA  = '/content/drive/MyDrive/Wav2Lip/person.mp4'

sys.argv = [
    'inference.py',
    '-s', IMAGEM_BASE,
    '-d', VIDEO_GUIA
]

from inference import main
main()